# D2 · Monthly recommendation queries and evaluation

What could we know at the start of a month, and which products were still valid purchases when the month closed? On CPU, run the cells in order. You will trace purchases to an October validation query, inspect the data shared by MF, LightGBM, LRURec and Mamba4Rec, and check the evaluator against hand calculations. The only rankings in this document use invented scores to test the arithmetic. The November test remains unopened. [Data protocol](../docs/data_protocol.md) · [D1 cleaning](project2_colab.ipynb)

# D2 · 月度推荐查询与评估

月初预测时能看到什么？月末哪些商品仍算作有效购买？选择 CPU 并顺序运行全部单元格。本文从发票追踪到十月验证查询，检查 MF、LightGBM、LRURec、Mamba4Rec 共用的数据，并用手算结果校验评价器。文中仅使用虚构分数检验指标算法；十一月测试保持封存。[数据协议](../docs/data_protocol.zh-CN.md) · [D1 清洗](project2_colab.ipynb)

## Open the project on Colab or in a local checkout

In Colab, authorize Google Drive. The default project folder is `My Drive/CS5481_Project2_OnlineRetail`, the same as D1. The next cell retrieves the exact Python helper files referenced by this notebook into temporary storage and verifies their SHA-256 digests. A project folder with the two D1 inputs and their manifest can be used directly. Otherwise the notebook downloads those three pinned files. The output folder always belongs to your Drive project. If you keep your project in another Drive folder, set `RETAIL_PROJECT_ROOT` to its path before running.

Locally, open a checkout with the D1 artifacts and install its requirements. From the repository root, the equivalent command to build D2 is `python -m scripts.build_queries --project-root .`. Running the notebook reuses a matching D2 manifest when present. [Source and dependencies](../requirements.txt)

## 在 Colab 或本地仓库中打开项目

在 Colab 授权挂载 Google Drive。默认项目目录与 D1 相同，为 `我的云端硬盘/CS5481_Project2_OnlineRetail`。下个单元格会把本 notebook 对应版本的 Python 源码下载到临时目录，逐一核对 SHA-256。若项目目录已有两份 D1 输入和清单，就直接使用；否则下载固定版本的这三份文件。D2 输出保存在自己的 Drive 项目中。如项目放在 Drive 的其他位置，可在运行前通过 `RETAIL_PROJECT_ROOT` 指定目录。

本地打开包含 D1 文件的仓库，按依赖文件安装环境。在仓库根目录执行 `python -m scripts.build_queries --project-root .` 也可生成 D2。已有相符的 D2 清单时，运行本 notebook 会复用现有产物。[源码依赖](../requirements.txt)

In [1]:
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
REVISION = "b46a0f66faee2fd822525bbde5e6783ef1635576"
RAW_URL = f"https://raw.githubusercontent.com/NobunagaK/online-retail-top10-recommendation/{REVISION}"
HELPERS = {
    "scripts/build_queries.py": "ab24ed5c9354feee652f2060bbe830ded13fff626d8bd1b3cf285336db3d404a",
    "src/data/clean.py": "1366b16d892299f55e7a31224f50c7190fb8fec9c725b42defcb7db04a67a528",
    "src/data/monthly_queries.py": "7a46bebb34090990fb1a5bc9a0124d4c767a70a2ec619295b7eae3a3a7444c08",
    "src/evaluation/ranking.py": "68cf1877997a8af9a3b3f18a1adf9aecbc06918f7ed0123a0c5ad9006df33570",
    "src/data/history_features.py": "5d953a40fb8c674f475f0b8ac1c961cb81d815770f15a5f3d973a8234eebf512",
    "src/data/sequences.py": "10641272a94ea1fd9b2559ce23f39edea7f5bbaf3f9937deff3047b56453d0e8",
    "src/data/purchase_snapshot.py": "a79a963c173b8f7e2f5e074a3ceac97236012f42444ca72000d0d86f6ee013c4",
    "src/data/baskets.py": "5c0d11dd80f326903e85b83f769ceacea13a25b289ba06109ce7dbca36ebef82"
}
D1_MANIFEST = "results/data_manifest.json"
D1_INPUTS = ("data/processed/signed_ledger_b.csv.gz", "results/credit_allocations.csv.gz")
D1_MANIFEST_SHA256 = "57ef1484a901f5c6a8f410bc572b85c9cc0c7d56209171c10f71620bb2736849"


def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def fetch(relative, folder, expected_sha):
    path = folder / relative
    if not path.is_file() or sha256(path) != expected_sha:
        path.parent.mkdir(parents=True, exist_ok=True)
        candidate = path.with_suffix(path.suffix + ".download")
        urlretrieve(f"{RAW_URL}/{relative}", candidate)
        if sha256(candidate) != expected_sha:
            raise ValueError(f"Downloaded bytes did not match the pinned SHA-256: {relative}")
        candidate.replace(path)
    return path


if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    project = Path(os.environ.get("RETAIL_PROJECT_ROOT", "/content/drive/MyDrive/CS5481_Project2_OnlineRetail"))
    source_root = Path("/content/retail_d2") / REVISION
    for name, expected in HELPERS.items():
        fetch(name, source_root, expected)
    if importlib.util.find_spec("pyarrow") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow==20.0.0"], check=True)
else:
    start = Path(os.environ.get("RETAIL_PROJECT_ROOT", Path.cwd())).resolve()
    project = next(p for p in (start, *start.parents) if (p / "scripts/build_queries.py").is_file())
    source_root = project
    different = [name for name, expected in HELPERS.items() if not (project/name).is_file() or sha256(project/name) != expected]
    if different:
        raise ValueError(f"The notebook expects helper revision {REVISION}. Check these files: {different}")

sys.path.insert(0, str(source_root))
import pandas as pd
import pyarrow
from IPython.display import display
from scripts.build_queries import run, verify_outputs
from src.data.purchase_snapshot import purchase_line_balances
from src.data.sequences import prepare_sequence
from src.evaluation.ranking import evaluate_query, sample_auc_pairs, sampled_auc, summarize_metrics

## Verify the signed ledger and the credit allocations

D2 reads the known-customer ledger (`signed_ledger_b.csv.gz`) and the D1 credit-to-sale links (`credit_allocations.csv.gz`). Each digest must match the pinned net-retained D1 manifest. A matching ledger is necessary because a credit link refers to a specific source sale row.

## 核对已知客户账本和贷项分配

D2 使用已知客户账本 `signed_ledger_b.csv.gz` 以及把贷项归到原单的 `credit_allocations.csv.gz`。两个文件都必须与固定净保留版 D1 清单中的校验值相符，因为每个贷项链接都指向特定原始销售行。

In [2]:
has_manifest = (project/D1_MANIFEST).is_file() and sha256(project/D1_MANIFEST) == D1_MANIFEST_SHA256
local_manifest = json.loads((project/D1_MANIFEST).read_text()) if has_manifest else None
local_inputs_match = has_manifest and all(
    (project/p).is_file() and sha256(project/p) == local_manifest["outputs"][p]["sha256"] for p in D1_INPUTS
)
data_root = project
if not local_inputs_match:
    if not COLAB:
        raise FileNotFoundError("Run D1 cleaning in this project or use the matching checkout.")
    data_root = source_root
    fetch(D1_MANIFEST, data_root, D1_MANIFEST_SHA256)
    d1 = json.loads((data_root/D1_MANIFEST).read_text())
    for name in D1_INPUTS:
        fetch(name, data_root, d1["outputs"][name]["sha256"])
d1 = json.loads((data_root/D1_MANIFEST).read_text())
assert d1["recommendation_quantity_policy"] == "net_retained_all_matched_credits_v1"
print("Verified helper revision / 已核对源码版本", REVISION)
print("pandas", pd.__version__, " / PyArrow", pyarrow.__version__)

Verified helper revision / 已核对源码版本 b46a0f66faee2fd822525bbde5e6783ef1635576
pandas 2.2.3  / PyArrow 20.0.0


## Build or verify the month-end files

For each training month from February through September, construct history and catalog from completed baskets and linked credits strictly before the month starts. Build labels from purchases in [month start, next month start), subtracting only credits known before that month's end. October validation uses history before 1 October and labels before 1 November. Use each package’s manifest to check every output before reading it. AUC sampling uses seed 5481 independently of the training negative seed 42. [Monthly query builder](../scripts/build_queries.py)

## 生成或核对月末文件

二月至九月的每个训练月份，都在月初用严格早于该时刻的已完成购物篮和已匹配贷项构造历史与候选目录。标签选择 [月初, 次月初) 内的购买，只扣减严格早于次月初的贷项。十月验证使用十月一日前的历史和十一月一日前的标签。读取数据前按清单核对每个输出文件。AUC 使用固定种子 5481，训练负例另用种子 42。[月度查询构建程序](../scripts/build_queries.py)

In [3]:
d2 = Path(os.environ.get("RETAIL_D2_OUTPUT", project/"data/processed/d2"))
manifest_path = d2/"manifest.json"
expected_inputs = {path:sha256(data_root/path) for path in (D1_MANIFEST, *D1_INPUTS)}
expected = {
    "protocol":"v3.4_monthly_net_retained_v1", "recommendation_quantity_policy":"net_retained_all_matched_credits_v1",
    "input_sha256":expected_inputs, "train_negative_seed":42, "auc_sample_seed":5481, "splits":["train","validation"],
    "frozen_evaluation_history_end":"2011-10-01", "last_target_end_exclusive":"2011-11-01",
    "max_training_negatives_per_positive":5,"max_auc_negatives_per_positive":10,"shared_initial_sequence_length":64,
}
tables_required = {"queries","labels","catalogs","item_vocab","item_support","histories","customer_features",
                   "item_features","customer_item_features","train_pairs","auc_pairs","cohorts"}
expected_outputs = {"coverage.csv", *[f"{name}.parquet" for name in tables_required]}

def check_protocol(candidate):
    for key, value in expected.items():
        if candidate.get(key) != value:
            raise ValueError(f"D2 protocol setting differs: {key}")
    if set(candidate["outputs"]) != expected_outputs:
        raise ValueError("D2 output files differ from the protocol.")

try:
    manifest = verify_outputs(d2)
    check_protocol(manifest)
    print("Verified stored D2 files / 已核对已有 D2 文件")
except (FileNotFoundError, ValueError, KeyError, TypeError):
    print("Building D2 from verified D1 inputs / 用已核对的 D1 输入重建 D2")
    manifest = run(data_root, output_dir=d2, seed=42)
    check_protocol(manifest)
display(pd.DataFrame([{"file":name,"rows":meta["rows"]} for name,meta in manifest["outputs"].items()]))
print("D2 protocol / 数据协议", manifest["protocol"])
print("Output folder within the project / 项目中的结果目录", "data/processed/d2" if d2 == project/"data/processed/d2" else d2.name)

Building D2 from verified D1 inputs / 用已核对的 D1 输入重建 D2


,file,rows
0,train_pairs.parquet,843720
1,histories.parquet,23252
2,customer_features.parquet,23252
3,item_features.parquet,27783
4,customer_item_features.parquet,1005341
5,catalogs.parquet,27783
6,cohorts.parquet,25921
7,queries.parquet,6188
8,labels.parquet,246998
9,coverage.csv,9


D2 protocol / 数据协议 v3.4_monthly_net_retained_v1
Output folder within the project / 项目中的结果目录 data/processed/d2


## Find the customers and products that each query can cover

A training query needs prior history and an in-catalog target purchase. For validation, the customer must also have appeared in a valid training query. Previously purchased products remain candidates. Primary metrics use eligible queries and their distinct in-catalog monthly positives. Supplementary Recall covers training-query customers with any purchase that month and keeps all distinct purchased SKUs in its denominator, including customers whose purchases are all outside the catalog. The eight training months contain **5,430** eligible queries. October has 1,357 active customers and **758** qualifying for the primary metrics, with 25,051 in-catalog positive pairs. One further training-query customer buys only out-of-catalog items, so 759 customers enter supplementary Recall. Read the coverage table separately from ranking performance.

## 查看每个月能覆盖的客户和商品

一条训练查询需要有历史，且目标月购买过至少一种当时目录内商品。验证客户还须出现过有效训练查询。买过的商品仍可推荐。主指标使用符合资格的查询和去重的月内目录正例。补充 Recall 面向目标月有购买的训练查询客户，分母包含其所有去重购买商品，包括只购买目录外商品的客户。八个训练月共有 **5,430** 条有效查询。十月有 1,357 名活跃客户，其中 **758** 人进入主指标，共计 25,051 个目录内正例。另有一位曾进入训练查询的客户只买了目录外商品，故补充 Recall 的客户总数为 759。覆盖统计与后续模型排名成绩分别解释。

In [4]:
def table(name, **kwargs):
    return pd.read_parquet(d2/f"{name}.parquet", **kwargs)

queries = table("queries")
catalogs = table("catalogs")
labels = table("labels")
coverage = pd.read_csv(d2/"coverage.csv")
display(coverage[["split","target_start","catalog_items","active_customers","eligible_customers","eligible_share_of_active",
                  "out_of_catalog_positive_pairs","supplemental_customers"]])
assert set(queries["split"]) == {"train","validation"}
assert queries["target_start"].max() == pd.Timestamp("2011-10-01")
assert queries["target_end"].max() == pd.Timestamp("2011-11-01")
assert queries["query_id"].is_unique
print("Unique eligible customer–month queries / 可评估客户–月份查询数", len(queries))

,split,target_start,catalog_items,active_customers,eligible_customers,eligible_share_of_active,out_of_catalog_positive_pairs,supplemental_customers
0,train,2011-02-01,2621,756,376,0.497354,412,0
1,train,2011-03-01,2748,971,517,0.532441,869,0
2,train,2011-04-01,2871,851,549,0.645123,646,0
3,train,2011-05-01,2980,1053,767,0.728395,2310,0
4,train,2011-06-01,3102,987,744,0.753799,1716,0
5,train,2011-07-01,3260,945,758,0.802116,481,0
6,train,2011-08-01,3325,933,762,0.816720,470,0
7,train,2011-09-01,3381,1257,957,0.761337,1887,0
8,validation,2011-10-01,3495,1357,758,0.558585,1787,759


Unique eligible customer–month queries / 可评估客户–月份查询数 6188


## Trace one October query from the month-start catalog to month-end purchases

Customer 17652 entered October with **179 basket-item events in seven invoices**, ending on 16 September. In October this customer bought three units of SKU 23494 on invoice 569329, returned all three through C569979, and bought three again on invoice 573104. The surviving second purchase yields one positive monthly label. Two other October purchases, SKUs 23581 and 23583, fall outside the October 1 catalog. We also inspect a full return with no other surviving October purchase (15038 / 23250) and a partial return (17811 / 22961, 24 − 10 = 14). The row numbers refer to the original workbook.

The D1 accounting ledger and allocations contain transaction records. Read only the columns needed for these customers and retain records earlier than 1 November for this demonstration. Compute balances at both boundaries with the production helper, then check the persisted D2 labels. Credit timestamps on or after the boundary cannot enter either snapshot.

## 从月初目录追溯到月底购买标签

客户 17652 进入十月时，历史中有 **7 张发票、179 个发票–商品事件**，最后一笔在 9 月 16 日。十月在发票 569329 买 3 件商品 23494，随后通过 C569979 全退，又通过 573104 买回 3 件。因此留下一个月度正例。该客户十月购买的另两种商品 23581、23583 在十月一日的候选目录之外。再对照全退后当月没有其他有效购买（15038 / 23250）和部分退货（17811 / 22961，24 − 10 = 14）。行号可在原始工作簿中定位。

D1 带符号账本和配对表按原始交易记录存储。下方为演示只保留这几位客户在十一月一日以前的明细，用正式函数分别回放月初和月末，再核对已保存的 D2 标签。在截止时刻或其后出现的贷项不进入对应快照。

In [5]:
start, end = pd.Timestamp("2011-10-01"), pd.Timestamp("2011-11-01")
people = {"15038","17652","17811"}
def observed_rows(filename, date_column):
    selected = []
    columns = (["ExcelRow",*d1["source"]["raw_fields"],"ledger_type"] if filename == D1_INPUTS[0]
               else ["CustomerID","credit_excel_row","credit_invoice_no","credit_time","purchase_excel_row","credited_units","status"])
    identifiers = {c:"string" for c in columns if c in {"InvoiceNo","CustomerID","StockCode","credit_invoice_no"}}
    for chunk in pd.read_csv(data_root/filename, usecols=columns, chunksize=50_000,
                             keep_default_na=False, na_values=[""], dtype=identifiers):
        dates = pd.to_datetime(chunk[date_column])
        selected.append(chunk.loc[chunk["CustomerID"].isin(people) & dates.lt(end)].copy())
    return pd.concat(selected, ignore_index=True)

sales = observed_rows(D1_INPUTS[0], "InvoiceDate")
credits = observed_rows(D1_INPUTS[1], "credit_time")
oct_start = purchase_line_balances(sales, credits, cutoff=start)
oct_end = purchase_line_balances(sales, credits, cutoff=end)
source_rows = [370990,371018,373424,422680]
examples = oct_end.loc[oct_end["ExcelRow"].isin(source_rows),
                       ["ExcelRow","CustomerID","StockCode","InvoiceNo","InvoiceDate","original_quantity","matched_credit_units","Quantity"]].copy()
examples["in_history_at_month_start"] = examples["ExcelRow"].isin(oct_start["ExcelRow"])
oct_query_ids = {x:f"2011-10/{x}" for x in people}
oct_labels = labels.loc[labels["query_id"].isin(oct_query_ids.values())]
target_keys = set(zip(oct_labels["query_id"],oct_labels["StockCode"]))
examples["month_end_positive"] = [(oct_query_ids[c],s) in target_keys for c,s in zip(examples.CustomerID,examples.StockCode)]
display(credits.loc[credits["purchase_excel_row"].isin(source_rows),
                    ["credit_excel_row","credit_invoice_no","credit_time","purchase_excel_row","credited_units","status"]])
display(examples)
assert examples.set_index("ExcelRow")["Quantity"].to_dict() == {370990:0,371018:14,373424:0,422680:3}
assert examples.set_index("ExcelRow")["month_end_positive"].to_dict() == {370990:False,371018:True,373424:True,422680:True}
assert not examples["in_history_at_month_start"].any()

,credit_excel_row,credit_invoice_no,credit_time,purchase_excel_row,credited_units,status
17,382918,C569979,2011-10-06 19:29:00,373424.0,3,return
22,414261,C572440,2011-10-24 12:06:00,371018.0,10,return
27,422616,C573097,2011-10-27 14:42:00,370990.0,6,return


,ExcelRow,CustomerID,StockCode,InvoiceNo,InvoiceDate,original_quantity,matched_credit_units,Quantity,in_history_at_month_start,month_end_positive
812,370990,15038,23250,569206,2011-10-02 10:58:00,6,6,0,False,False
840,371018,17811,22961,569207,2011-10-02 11:04:00,24,10,14,False,True
861,373424,17652,23494,569329,2011-10-03 13:03:00,3,3,0,False,True
948,422680,17652,23494,573104,2011-10-27 14:59:00,3,0,3,False,True


## Intersect monthly purchases with the catalog

A positive sale can be outside the candidates available at prediction time. The following query shows 44 candidate positives from 46 distinct valid October items. Both excluded SKUs remain visible in the full-item coverage count.

## 将月内购买与候选目录取交集

有效购买的商品可能不在预测时已有的目录中。下面这条查询十月共买了 46 种不同商品，目录内正例为 44 个。两个目录外商品继续计入所有商品口径的覆盖统计。

In [6]:
example_query = oct_query_ids["17652"]
display(queries.loc[queries["query_id"].eq(example_query),
                    ["query_id","history_end","target_start","target_end","positive_count","full_positive_count","catalog_id"]])
customer_labels = oct_labels.loc[oct_labels["query_id"].eq(example_query)]
outside = customer_labels.loc[~customer_labels["in_catalog"]]
display(outside[["query_id","StockCode","item_id","in_catalog"]])
assert set(outside["StockCode"]) == {"23581","23583"}
assert customer_labels.loc[customer_labels["StockCode"].eq("23494"),"in_catalog"].tolist() == [True]

,query_id,history_end,target_start,target_end,positive_count,full_positive_count,catalog_id
6117,2011-10/17652,2011-10-01,2011-10-01,2011-11-01,44,46,2011-10-01


,query_id,StockCode,item_id,in_catalog
243067,2011-10/17652,23581,<NA>,False
243068,2011-10/17652,23583,<NA>,False


## Keep the same history for October validation and November forecasting

Read the October 1 history for customer 17652 once. MF and LightGBM can use counts and purchase recency; the sequence models read the same item identities in basket order. Sort within a basket deterministically, retain repeats across baskets, reserve 0 for padding, and keep the most recent 64 events for the common baseline. Padding a short left- or right-aligned history gives the same unpadded items. The last valid array index can differ. For this customer, the full history has 179 events; the 64-event suffix is used for either model.

Reuse exactly this October 1 history and these 7-, 30- and 90-day windows for November. The feature `days_since_last_purchase` is anchored to the history cutoff and stays constant. A separately named calendar lag measured from the *target* start increases by 31 days. The upcoming target date is known in advance. This comparison needs no November purchases, customers or labels.

## 十月验证和十一月预测使用同一份历史

只读取一次客户 17652 在十月一日的历史。MF、LightGBM 可使用计次与最近购买间隔；两种序列模型按相同的购物篮顺序读取同一批商品。篮内排序固定，跨篮复购保留，0 为填充，公共基线取最近 64 个事件。对短历史分别左填充和右填充，去掉 0 后得到相同商品，最后有效位置的数组下标可能不同。该客户原历史有 179 个事件，两模型统一取末尾 64 个。

十一月沿用这份十月一日历史和以该日为锚点的 7、30、90 天窗口。特征 `days_since_last_purchase` 相对历史截止点计算，数值相同；另列的“相对目标月初的天数”随日历向前增加 31 天。预测前已知目标日期，核验无需打开十一月购买数据、客户名单或标签。

In [7]:
lookup = [("customer_id","=","17652"),("history_end","=",start)]
history = table("histories", filters=lookup).iloc[0]
feat = table("customer_features", filters=lookup).iloc[0]
sequence = [int(i) for i in history["history_item_ids"]]
assert len(sequence) == history["history_len"] == 179
assert feat["invoice_count"] == 7 and feat["last_purchase_time"] < start
vocab = table("item_vocab").set_index("item_id")["stock_code"]
sequence_views = []
for padding in ("left","right"):
    for ids in (sequence[-3:],sequence):
        padded, n, last = prepare_sequence(ids, max_length=64, padding=padding)
        assert [x for x in padded if x] == ids[-64:] and padded[last] == ids[-1]
        sequence_views.append({"padding":padding,"original_events":len(ids),"used":n,"last_valid_index":last,
                               "last_3_products":[vocab.loc[i] for i in ids[-3:]]})
display(pd.DataFrame(sequence_views))
recency = []
for target in (start,end):
    recency.append({"target_start":target,"history_end":start,"last_purchase_time":feat["last_purchase_time"],
                    "history_lag_days":feat["days_since_last_purchase"],
                    "target_calendar_lag_days":(target-feat["last_purchase_time"])/pd.Timedelta(days=1),
                    **{f"invoices_{d}d":int(feat[f"invoices_{d}d"]) for d in (7,30,90)}})
recency = pd.DataFrame(recency)
display(recency)
assert recency["history_lag_days"].nunique() == 1
assert recency["target_calendar_lag_days"].diff().iloc[-1] == 31
# Excluding October transactions before replay and filtering them at replay yield the same October 1 customer histories.
earlier = sales.loc[pd.to_datetime(sales["InvoiceDate"]).lt(start)]
earlier_credits = credits.loc[pd.to_datetime(credits["credit_time"]).lt(start)]
early_replay = purchase_line_balances(earlier, earlier_credits, cutoff=start)
pd.testing.assert_frame_equal(oct_start.reset_index(drop=True), early_replay.reset_index(drop=True))

,padding,original_events,used,last_valid_index,last_3_products
0,left,3,3,63,"[47590A, 85099B, 85123A]"
1,left,179,64,63,"[47590A, 85099B, 85123A]"
2,right,3,3,2,"[47590A, 85099B, 85123A]"
3,right,179,64,63,"[47590A, 85099B, 85123A]"


,target_start,history_end,last_purchase_time,history_lag_days,target_calendar_lag_days,invoices_7d,invoices_30d,invoices_90d
0,2011-10-01,2011-10-01,2011-09-16 14:39:00,14.389583,14.389583,0,1,2
1,2011-11-01,2011-10-01,2011-09-16 14:39:00,14.389583,45.389583,0,1,2


## Inspect training negatives and validation AUC samples

MF and LightGBM use the same distinct training negatives per customer-month, sampled without replacement from eligible unpurchased catalog items. Keep all positives and at most five negatives per positive (seed 42). Validation ranks the full candidate catalog. Its supplementary AUC samples at most ten negatives per in-catalog positive (seed 5481) and retains every positive. An unobserved purchase is a negative for these calculations. The data do not establish customer dislike.

October's 3,495 candidates include 114 with neither an MF training-pair appearance nor a sequence training catalog appearance. Keep these items scoreable. The flags show opportunities for a training signal. Later model implementations must track which parameter rows actually received one.

## 检查训练负例与验证 AUC 抽样

MF 和 LightGBM 共用每个客户–月份的训练负例，从当时目录中当月未购商品无放回抽取。保留所有正例，每个正例最多对应 5 个负例（种子 42）。验证排名遍历全部候选。补充 AUC 保留所有目录正例，每个正例最多抽 10 个负例（种子 5481）。这里的负例表示没有观察到购买，无法据此判定客户不喜欢该商品。

十月有 3,495 个候选商品，其中 114 个既未出现在 MF 训练配对中，也未出现在序列模型训练目录中，仍保留评分资格。这两个标记衡量数据中是否存在获得训练信号的机会；各模型实现时仍须记录实际获得过训练信号的参数行。

In [8]:
train_id = queries.loc[queries["split"].eq("train"),"query_id"].iloc[0]
train_sample = table("train_pairs", filters=[("query_id","=",train_id)])
auc_sample = table("auc_pairs", filters=[("query_id","=",example_query)])
for qid, frame, ratio in [(train_id,train_sample,5),(example_query,auc_sample,10)]:
    in_catalog = labels.loc[labels["query_id"].eq(qid) & labels["in_catalog"],"item_id"]
    pos = set(frame.loc[frame["label"].eq(1),"item_id"])
    neg = set(frame.loc[frame["label"].eq(0),"item_id"])
    assert pos == set(in_catalog) and pos.isdisjoint(neg) and len(neg) <= ratio * len(pos)
    assert frame["item_id"].is_unique
    print(qid, "positive / 正例",len(pos)," · sampled negative / 抽样负例",len(neg))
catalog_id = queries.loc[queries["query_id"].eq(example_query),"catalog_id"].iloc[0]
visible = catalogs.loc[catalogs["catalog_id"].eq(catalog_id),["item_id","StockCode"]]
assert len(visible) == 3495
support = table("item_support")
candidate_support = visible.merge(support[["item_id","mf_observed_train_pair","sequence_in_train_catalog"]],on="item_id", validate="one_to_one")
support_counts = (candidate_support.groupby(["mf_observed_train_pair","sequence_in_train_catalog"])
                  .size().rename("october_candidate_items").reset_index())
display(support_counts)
assert len(candidate_support) == len(visible)
print("Full candidates for October validation / 十月验证完整候选数",len(visible))

2011-02/12359 positive / 正例 80  · sampled negative / 抽样负例 400
2011-10/17652 positive / 正例 44  · sampled negative / 抽样负例 440


,mf_observed_train_pair,sequence_in_train_catalog,october_candidate_items
0,False,False,114
1,True,True,3381


Full candidates for October validation / 十月验证完整候选数 3495


## Check Recall@10, NDCG@10 and sampled query-macro ROC AUC by hand

Suppose the relevant set is {B,D} and the first ten items are A,B,C,E,F,G,H,I,J,K. B is the only hit and appears at rank 2. Recall@10 is 1/2. Binary NDCG@10 divides the gain `1/log₂(3)` by the ideal gain `1 + 1/log₂(3)`, giving about 0.38685. If a third valid purchase is outside the catalog, supplementary Recall is 1/3. Score the 11-item catalog twice, once whole and once in blocks; both orders must agree.

For sampled AUC, compare raw scores within each query (win = 1, tie = 0.5). A positive at 0.8 versus negatives at 0.8 and 0.2 gives 0.75. A positive at 0.1 versus a negative at 0.3 gives zero. Equal weighting across these two queries gives 0.375. A query containing only positives or only negatives has undefined AUC and is counted separately. Ranking metrics and sampled AUC have different denominators. [Evaluator source](../src/evaluation/ranking.py)

## 手算 Recall@10、NDCG@10 和按查询宏平均的抽样 ROC AUC

设正例集合为 {B,D}，前十名依次是 A、B、C、E、F、G、H、I、J、K。只命中排在第 2 位的 B，故 Recall@10 为 1/2。二元 NDCG@10 用 `1/log₂(3)` 除以理想收益 `1 + 1/log₂(3)`，约为 0.38685。若另有一个目录外有效购买，补充 Recall 为 1/3。对全部 11 项分别整批和分块评分，两次输出应一致。

抽样 AUC 在每个查询内比较正负例原始分数，正例胜出计 1、平局计 0.5。正例 0.8 对负例 0.8 和 0.2，得到 0.75；正例 0.1 对负例 0.3，得到 0。两个查询等权平均为 0.375。仅有正例或仅有负例时 AUC 未定义，单独统计。排名指标和抽样 AUC 有各自的分母。[评价器源码](../src/evaluation/ranking.py)

In [9]:
from math import isclose, log2

items = pd.DataFrame({"item_id":range(1,12),"StockCode":list("ABCDEFGHIJK")})
ranking_order = list("ABCEFGHIJKD")
scores = items[["item_id"]].assign(score=[11-ranking_order.index(code) for code in items.StockCode])
pos_ids = set(items.loc[items["StockCode"].isin(["B","D"]),"item_id"])
sample = sample_auc_pairs("validation","toy-ranking",items,pos_ids,seed=5481)
whole, raw_scores, metrics = evaluate_query(items,scores,pos_ids,sample,all_positive_count=3)
blocks, raw_blocks, block_metrics = evaluate_query(items,[scores.iloc[:4],scores.iloc[4:]],pos_ids,sample,all_positive_count=3)
pd.testing.assert_frame_equal(whole,blocks)
pd.testing.assert_frame_equal(raw_scores,raw_blocks)
assert metrics == block_metrics and whole["StockCode"].tolist() == ranking_order[:10]
assert metrics["recall_at_10"] == 0.5 and isclose(metrics["ndcg_at_10"], (1/log2(3))/(1+1/log2(3)))
assert metrics["supplementary_recall_at_10"] == 1/3
display(whole[["rank","StockCode","score"]])
display(pd.Series({k:metrics[k] for k in ("recall_at_10","ndcg_at_10","supplementary_recall_at_10")},name="hand_check").to_frame())

,rank,StockCode,score
0,1,A,11.0
1,2,B,10.0
2,3,C,9.0
3,4,E,8.0
4,5,F,7.0
5,6,G,6.0
6,7,H,5.0
7,8,I,4.0
8,9,J,3.0
9,10,K,2.0


,hand_check
recall_at_10,0.500000
ndcg_at_10,0.386853
supplementary_recall_at_10,0.333333


## Check that ties and single-class queries are counted correctly

The first two queries contribute equally to the mean. The third has one positive and no negatives. Keep it in the count of skipped AUC queries and print the denominator used by the evaluator.

## 检验平局与单类别查询的处理

前两个查询等权参与平均。第三个查询只有正例，没有负例，计入跳过的 AUC 查询数。下面同时展示评价器实际使用的分母。

In [10]:
auc_examples = [([0.8],[0.8,0.2]),([0.1],[0.3]),([0.6],[])]
auc_metrics = []
for n,(p,n_scores) in enumerate(auc_examples,start=1):
    values = p + n_scores
    toy = pd.DataFrame({"item_id":range(1,len(values)+1),"StockCode":[str(i) for i in range(1,len(values)+1)]})
    yes = set(toy["item_id"].iloc[:len(p)])
    pairs = sample_auc_pairs("validation",f"toy-auc-{n}",toy,yes)
    _,_,m = evaluate_query(toy,toy[["item_id"]].assign(score=values),yes,pairs)
    assert m["sampled_auc"] == sampled_auc(p,n_scores)
    auc_metrics.append(m)
summary = summarize_metrics(auc_metrics)
display(pd.DataFrame(auc_metrics)[["auc_positive_items","auc_negative_items","sampled_auc"]])
display(pd.Series({k:summary[k] for k in ("sampled_auc","auc_valid_queries","auc_skipped_queries")},name="query_macro").to_frame())
assert summary["sampled_auc"] == 0.375 and summary["auc_valid_queries"] == 2 and summary["auc_skipped_queries"] == 1

,auc_positive_items,auc_negative_items,sampled_auc
0,1,2,0.75
1,1,1,0.00
2,1,0,NaN


,query_macro
sampled_auc,0.375
auc_valid_queries,2.000
auc_skipped_queries,1.000


## Interpret a frozen target at the end of the observation period

A credit first seen after a month closes may retrospectively reduce a frozen positive to zero. D2 evaluates what was on the books at the stated cutoff. The independent February–October audit finds **758 / 246,998** frozen distinct month–customer–SKU positives later disappeared by the workbook end (0.307%, before catalog and customer restrictions). October alone accounts for **105 / 43,978** (0.239%). The team-provided **788 / 301,922 ≈ 0.26%** figure spans February–November and includes the separately reported November aggregate. This notebook uses no November test labels to compute or demonstrate these rates. December observation stops on **9 December 2011 at 12:50**, so additional later credits remain unobservable.

Those percentages concern individual positive labels. Their effect on Recall or NDCG also depends on which queries and ranks are affected; sampled AUC depends on score ordering and its negative sample. A low aggregate rate cannot establish the absence or size of a metric change. No model has been scored here. The next modeling stage can train and validate MF on the fixed D2 files. [Pre-test audit rows](../results/d2_audit/label_delay_pretest.csv) · [Data protocol and limitations](../docs/data_protocol.md)

## 观察期结束时应怎样解读冻结的标签

贷项可能在目标月结束后才出现，使已经冻结的正例在事后回看时变成零。D2 评估的是规定截止点的账面状态。独立的二月至十月审计发现，在目录和客户资格筛选前，冻结的“月份–客户–商品”去重正例有 **758 / 246,998** 在工作簿结束前归零（0.307%），其中十月为 **105 / 43,978**（0.239%）。项目组提供的 **788 / 301,922 ≈ 0.26%** 覆盖二月至十一月，包含另行给出的十一月汇总。本文未使用十一月测试标签计算或展示这些比例。数据止于 **2011 年 12 月 9 日 12:50**，此后仍可能有未观察到的贷项。

以上比例的单位是正标签。对 Recall、NDCG 的具体影响取决于哪些查询及哪些名次受到影响；抽样 AUC 还取决于分数排序和抽样负例。较小的标签比例本身无法证明指标有无变化或变化多大。本文没有给项目模型打分；下一阶段可用固定 D2 文件训练、验证 MF。[测试前逐月核验结果](../results/d2_audit/label_delay_pretest.csv) · [数据协议与限制](../docs/data_protocol.zh-CN.md)